# Model Optimization: Quantization

In this notebook, we'll apply quantization techniques to our models using distributed processing. Instead of running the quantization on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the quantization on more powerful instances.

This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor
from sagemaker.pytorch.processing import PyTorchProcessor

# Import our utility functions for distributed processing
from sagemaker_processing import run_quantization_job

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics from file
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

# Load model information from file
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded information for {len(model_info)} models")

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create and Upload Quantization Script to S3

In [ ]:
# Create a quantization script
quantization_script = """
import os
import json
import torch
import argparse
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM

def load_model_and_tokenizer(model_name, task):
    """Load model and tokenizer based on task."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return model, tokenizer

def prepare_inputs(task, tokenizer, sample_input):
    """Prepare inputs for different model tasks."""
    if task == "sequence-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "token-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "question-answering":
        inputs = tokenizer(
            sample_input["question"],
            sample_input["context"],
            return_tensors="pt"
        )
    elif task == "masked-lm":
        inputs = tokenizer(sample_input, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return inputs

def measure_inference_time(model, inputs, num_runs=10):
    """Measure inference time for a model."""
    # Warm-up run
    with torch.no_grad():
        _ = model(**inputs)
    
    # Measure inference time
    start_time = torch.cuda.Event(enable_timing=True)
    end_time = torch.cuda.Event(enable_timing=True)
    
    timings = []
    with torch.no_grad():
        for _ in range(num_runs):
            start_time.record()
            _ = model(**inputs)
            end_time.record()
            torch.cuda.synchronize()
            timings.append(start_time.elapsed_time(end_time))
    
    return sum(timings) / len(timings)

def get_model_size(model):
    """Get model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

def apply_quantization(model, method="dynamic", bits=8):
    """Apply quantization to a model."""
    if method == "dynamic":
        # Dynamic quantization (quantizes weights at runtime)
        quantized_model = torch.quantization.quantize_dynamic(
            model, {torch.nn.Linear}, dtype=torch.qint8
        )
    elif method == "static":
        # Static quantization (requires calibration data)
        # This is a simplified version for demonstration
        model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
        torch.quantization.prepare(model, inplace=True)
        # Calibration would happen here with real data
        quantized_model = torch.quantization.convert(model, inplace=False)
    elif method == "aware":
        # Quantization-aware training (requires training)
        # This is a simplified version for demonstration
        model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
        torch.quantization.prepare_qat(model, inplace=True)
        # Training would happen here
        quantized_model = torch.quantization.convert(model, inplace=False)
    else:
        raise ValueError(f"Unsupported quantization method: {method}")
    
    return quantized_model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-info-path', type=str, required=True)
    parser.add_argument('--output-dir', type=str, required=True)
    parser.add_argument('--quantization-method', type=str, default='dynamic')
    parser.add_argument('--quantization-bits', type=int, default=8)
    args = parser.parse_args()
    
    # Load model info
    with open(args.model_info_path, 'r') as f:
        model_info = json.load(f)
    
    # Process each model
    quantized_metrics = {}
    for model_key, info in model_info.items():
        print(f"Processing {model_key}: {info['model_name']}")
        
        # Load model and tokenizer
        model, tokenizer = load_model_and_tokenizer(info['model_name'], info['task'])
        
        # Define sample input
        if info['task'] == 'sequence-classification':
            sample_input = "This is a sample input for sentiment analysis."
        elif info['task'] == 'token-classification':
            sample_input = "John Smith works at Microsoft in Seattle."
        elif info['task'] == 'question-answering':
            sample_input = {
                "question": "What is machine learning?",
                "context": "Machine learning is a branch of artificial intelligence."
            }
        elif info['task'] == 'masked-lm':
            sample_input = "The [MASK] is a large language model."
        
        # Prepare inputs
        inputs = prepare_inputs(info['task'], tokenizer, sample_input)
        
        # Apply quantization
        quantized_model = apply_quantization(
            model, 
            method=args.quantization_method,
            bits=args.quantization_bits
        )
        
        # Move to GPU if available
        if torch.cuda.is_available():
            model = model.to('cuda')
            quantized_model = quantized_model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
        
        # Measure metrics
        model_size = get_model_size(model)
        quantized_size = get_model_size(quantized_model)
        inference_time = measure_inference_time(model, inputs)
        quantized_inference_time = measure_inference_time(quantized_model, inputs)
        num_parameters = sum(p.numel() for p in model.parameters())
        quantized_parameters = sum(p.numel() for p in quantized_model.parameters())
        
        # Save quantized model
        output_dir = os.path.join(args.output_dir, model_key)
        os.makedirs(output_dir, exist_ok=True)
        torch.save(quantized_model.state_dict(), os.path.join(output_dir, "quantized_model.pt"))
        tokenizer.save_pretrained(output_dir)
        
        # Save metrics
        quantized_metrics[model_key] = {
            "model_key": model_key,
            "model_name": info['model_name'],
            "task": info['task'],
            "quantization_method": args.quantization_method,
            "quantization_bits": args.quantization_bits,
            "model_size": quantized_size,
            "original_size": model_size,
            "inference_time": quantized_inference_time,
            "original_inference_time": inference_time,
            "num_parameters": quantized_parameters,
            "size_reduction": (model_size - quantized_size) / model_size * 100,
            "speedup": inference_time / quantized_inference_time
        }
    
    # Save metrics to file
    with open(os.path.join(args.output_dir, 'quantized_metrics.json'), 'w') as f:
        json.dump(quantized_metrics, f, indent=2)

if __name__ == '__main__':
    main()
"""

# Write the script to a file
with open('quantization_script.py', 'w') as f:
    f.write(quantization_script)

# Upload the quantization script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'quantization_script.py', 
    S3_BUCKET, 
    'scripts/quantization_script.py'
)

print(f"Created and uploaded quantization script to s3://{S3_BUCKET}/scripts/quantization_script.py")

## 6. Launch Distributed Quantization Jobs

In [ ]:
# Define the instance type to use for quantization
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-quantization",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch quantization jobs for each model
quantization_jobs = {}

for model_key in model_info.keys():
    print(f"\nLaunching quantization job for {model_key}...")
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/scripts/quantization_script.py',
            destination='/opt/ml/processing/input/code/quantization_script.py'
        ),
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data/model_info.json'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{S3_BUCKET}/optimization/outputs/{model_key}'
        )
    ]
    
    # Run the processing job
    job = processor.run(
        code='quantization_script.py',
        inputs=inputs,
        outputs=outputs,
        arguments=[
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--quantization-method', 'dynamic',
            '--quantization-bits', '8'
        ]
    )
    
    # Store the job
    quantization_jobs[model_key] = job
    print(f"Launched quantization job: {job.job_name}")
    
    # Clean up temporary file
    os.remove(f'temp_{model_key}_info.json')

## 7. Monitor Job Status

In [ ]:
# Monitor job status
import time

# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Check job status every 30 seconds
all_completed = False
while not all_completed:
    all_completed = True
    job_statuses = {}
    
    for model_key, job in quantization_jobs.items():
        response = sagemaker_client.describe_processing_job(
            ProcessingJobName=job.job_name
        )
        status = response['ProcessingJobStatus']
        job_statuses[model_key] = status
        
        if status in ['InProgress', 'Stopping']:
            all_completed = False
    
    # Display status table
    status_df = pd.DataFrame({
        'Model': list(job_statuses.keys()),
        'Status': list(job_statuses.values())
    })
    display(status_df)
    
    if not all_completed:
        print("Waiting for jobs to complete...")
        time.sleep(30)
    else:
        print("All jobs completed!")

## 8. Collect Results

In [ ]:
# Download and combine results
quantized_metrics = {}

for model_key in model_info.keys():
    # Download metrics file
    try:
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}/quantized_metrics.json',
            f'temp_{model_key}_quantized_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_quantized_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        quantized_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('quantized_metrics.json', 'w') as f:
    json.dump(quantized_metrics, f, indent=2)

print(f"\nSaved quantized metrics for {len(quantized_metrics)} models to quantized_metrics.json")

## 9. Compare Results

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key in quantized_metrics.keys():
    if model_key in baseline_metrics:
        baseline = baseline_metrics[model_key]
        quantized = quantized_metrics[model_key]
        
        # Calculate improvements
        size_reduction = (baseline['model_size'] - quantized['model_size']) / baseline['model_size'] * 100
        time_reduction = (baseline['inference_time'] - quantized['inference_time']) / baseline['inference_time'] * 100
        
        comparison_data.append({
            'Model': quantized['model_name'],
            'Quantization Method': quantized['quantization_method'],
            'Bits': quantized['quantization_bits'],
            'Baseline Size (MB)': baseline['model_size'],
            'Quantized Size (MB)': quantized['model_size'],
            'Size Reduction (%)': size_reduction,
            'Baseline Inference (ms)': baseline['inference_time'],
            'Quantized Inference (ms)': quantized['inference_time'],
            'Inference Speedup (%)': time_reduction
        })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 10. Next Steps

Now that we've applied quantization to our models, we'll explore pruning techniques in the next notebook to further reduce model size.